<a href="https://colab.research.google.com/github/amosagekouassi-source/DI-Bootcamp/blob/master/Mini_Projet_W6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini Project: Sentiment Assistant with BERT Fine-Tuning

Welcome to the final day mini project! In this lab, you will fine-tune `bert-base-uncased` on movie reviews, evaluate the model, and connect the dots with real customer-support scenarios.

In [ ]:
!pip install -q tensorflow tensorflow-datasets transformers accelerate evaluate

## Imports & Hardware Check
We begin by confirming versions and hardware availability.

In [ ]:
import platform
import tensorflow as tf
import tensorflow_datasets as tfds
from transformers import BertTokenizer, TFBertForSequenceClassification
import numpy as np

print("Python version      :", platform.python_version())
print("TensorFlow version  :", tf.__version__)
print("GPU devices detected:", tf.config.list_physical_devices('GPU'))

## Load the IMDB Reviews Dataset
We use the IMDB dataset which contains 50,000 movie reviews (balanced positive/negative).

In [ ]:
(ds_train, ds_test), ds_info = tfds.load(
    "imdb_reviews",
    split=(tfds.Split.TRAIN, tfds.Split.TEST),
    as_supervised=True,
    with_info=True
)
print(ds_info)

In [ ]:
for text, label in ds_train.take(2):
    print("Label:", "Positive" if label.numpy() else "Negative")
    print(text.numpy().decode()[:250], "...\n")

## Tokenizer Setup & Data Pipeline
BERT requires specific formatting: Input IDs, Attention Masks, and Segment IDs.

In [ ]:
MAX_LENGTH = 256
BATCH_SIZE = 16

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased", do_lower_case=True)

def encode_review(review_input):
    if isinstance(review_input, bytes):
        review_text = review_input.decode("utf-8")
    elif hasattr(review_input, "numpy"):
        review_text = review_input.numpy().decode("utf-8")
    else:
        review_text = str(review_input)

    return tokenizer.encode_plus(
        review_text,
        add_special_tokens=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_token_type_ids=True,
    )

def tf_encode(text, label):
    encoded = tf.py_function(
        func=lambda t: list(encode_review(t).values()),
        inp=[text],
        Tout=[tf.int32, tf.int32, tf.int32]
    )
    return {
        "input_ids": encoded[0],
        "attention_mask": encoded[1],
        "token_type_ids": encoded[2]
    }, label

def prepare_dataset(dataset):
    return (
        dataset
        .map(tf_encode, num_parallel_calls=tf.data.AUTOTUNE)
        .shuffle(2000)
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )

train_ds = prepare_dataset(ds_train)
test_ds  = prepare_dataset(ds_test)

## Initialize the Fine-Tuning Model
Loading the pre-trained BERT model with a classification head.

In [ ]:
model = TFBertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2,
    use_safetensors=False
)

optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5, epsilon=1e-8)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metrics = [tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")]

model.compile(optimizer=optimizer, loss=loss_fn, metrics=metrics)
model.summary()

## Train and Monitor

In [ ]:
EPOCHS = 2
history = model.fit(
    train_ds,
    validation_data=test_ds.take(100), # Using a subset for validation speed
    epochs=EPOCHS
)

## Evaluate on the Held-Out Test Set

In [ ]:
eval_metrics = model.evaluate(test_ds)
print(f"Test Accuracy: {eval_metrics[1]:.4f}")

## Build a Reusable Inference Helper

In [ ]:
def predict_sentiment(text: str):
    inputs = encode_review(text)
    # Add batch dimension
    input_ids = tf.expand_dims(inputs['input_ids'], 0)
    attention_mask = tf.expand_dims(inputs['attention_mask'], 0)
    token_type_ids = tf.expand_dims(inputs['token_type_ids'], 0)

    outputs = model({
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "token_type_ids": token_type_ids
    })

    logits = outputs.logits
    probs = tf.nn.softmax(logits, axis=-1).numpy()[0]
    label_idx = np.argmax(probs)
    label = "Positive" if label_idx == 1 else "Negative"

    return label, float(probs.max())

custom_sentence = "The onboarding emails were confusing, but the agent fixed everything politely."
label, confidence = predict_sentiment(custom_sentence)
print(f"Prediction: {label} (confidence={confidence:.3f})")

## Reflection Questions

1. **What lever (data cleaning, hyperparameters, more epochs) most improved results?**
   *Answer here*

2. **Where would you add guardrails before deploying this sentiment signal live?**
   *Answer here*

3. **Which stakeholders benefit the most (support lead, product manager, compliance officer)?**
   *Answer here*